# 🧪 Capstone 9.3: Experiment Training

**Goal**: Train 4+ model types with full MLFlow tracking, then run HPO on the best model type.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
import optuna
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings; warnings.filterwarnings('ignore')

mlflow.autolog(disable=True)
mlflow.set_experiment("capstone-wine-quality")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("✅ Setup complete!")

## 1. Train Multiple Model Types (Baselines)

In [ ]:
def train_and_log(run_name, model, model_type, extra_params=None):
    """Train a model and log everything to MLFlow."""
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "stage": "baseline_training",
            "author": "sujat",
            "model_type": model_type,
        })
        
        # Log params
        params = model.get_params()
        # Filter to the most relevant params
        key_params = {k: v for k, v in params.items() if v is not None and k != 'random_state'}
        mlflow.log_params(key_params)
        if extra_params:
            mlflow.log_params(extra_params)
        
        # Train
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Metrics
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_weighted": f1_score(y_test, y_pred, average='weighted'),
            "precision_weighted": precision_score(y_test, y_pred, average='weighted'),
            "recall_weighted": recall_score(y_test, y_pred, average='weighted'),
        }
        mlflow.log_metrics(metrics)
        
        # Log model
        sig = infer_signature(X_train, y_pred)
        mlflow.sklearn.log_model(model, "model", signature=sig)
        
        print(f"  {run_name:.<40} acc={metrics['accuracy']:.4f}  f1={metrics['f1_weighted']:.4f}")
        return metrics

print("🔄 Training baselines...\n")

# Model 1: Decision Tree
train_and_log("baseline_decision_tree",
    DecisionTreeClassifier(max_depth=5, random_state=42), "DecisionTree")

# Model 2: Random Forest
train_and_log("baseline_random_forest",
    RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42), "RandomForest")

# Model 3: Gradient Boosting
train_and_log("baseline_gradient_boosting",
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42), "GradientBoosting")

# Model 4: Logistic Regression (with scaling)
train_and_log("baseline_logistic_regression",
    Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]), "LogisticRegression")

# Model 5: SVM (with scaling)
train_and_log("baseline_svm",
    Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', random_state=42))
    ]), "SVM")

print("\n✅ All baselines trained!")

## 2. Quick Comparison — Find Best Model Type

In [ ]:
runs = mlflow.search_runs(
    experiment_names=["capstone-wine-quality"],
    filter_string="tags.stage = 'baseline_training'",
    order_by=["metrics.accuracy DESC"]
)

cols = [c for c in ["tags.mlflow.runName", "tags.model_type", 
        "metrics.accuracy", "metrics.f1_weighted"] if c in runs.columns]

print("📊 Baseline Results:")
print(runs[cols].to_string(index=False))

best_type = runs.iloc[0].get("tags.model_type", "unknown")
print(f"\n🏆 Best model type: {best_type}")

## 3. HPO on Best Model Type (Nested Runs)

In [ ]:
def hpo_objective(trial):
    """Optuna objective for RandomForest HPO."""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }
    
    with mlflow.start_run(run_name=f"hpo_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.set_tag("stage", "hpo")
        
        model = RandomForestClassifier(**params, random_state=42)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
        
        mlflow.log_metric("cv_accuracy_mean", cv_scores.mean())
        mlflow.log_metric("cv_accuracy_std", cv_scores.std())
    
    return cv_scores.mean()

print("🔄 Running HPO (40 trials)...\n")

with mlflow.start_run(run_name="hpo_random_forest"):
    mlflow.set_tags({"stage": "hpo_parent", "author": "sujat", "model_type": "RandomForest"})
    mlflow.log_param("n_trials", 40)
    
    study = optuna.create_study(direction="maximize")
    study.optimize(hpo_objective, n_trials=40, show_progress_bar=True)
    
    mlflow.log_metric("best_cv_accuracy", study.best_value)
    for k, v in study.best_params.items():
        mlflow.log_param(f"best_{k}", v)
    
    hpo_run_id = mlflow.active_run().info.run_id

print(f"\n🏆 Best CV accuracy: {study.best_value:.4f}")
print(f"   Best params: {study.best_params}")

## 4. Train Final Model with Best HPO Params

In [ ]:
with mlflow.start_run(run_name="final_best_model"):
    mlflow.set_tags({"stage": "final_model", "author": "sujat", "model_type": "RandomForest"})
    mlflow.log_params(study.best_params)
    
    final_model = RandomForestClassifier(**study.best_params, random_state=42)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)
    
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_weighted": f1_score(y_test, y_pred, average='weighted'),
        "precision_weighted": precision_score(y_test, y_pred, average='weighted'),
        "recall_weighted": recall_score(y_test, y_pred, average='weighted'),
    }
    mlflow.log_metrics(metrics)
    
    # Log model with signature and input example
    sig = infer_signature(X_train, y_pred)
    mlflow.sklearn.log_model(
        final_model, "model",
        signature=sig,
        input_example=X_train[:3]
    )
    
    # Log confusion matrix
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    import os
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=wine.target_names, ax=ax, cmap='Blues'
    )
    plt.title('Final Model — Confusion Matrix')
    plt.tight_layout()
    fig.savefig("final_cm.png", dpi=150)
    mlflow.log_artifact("final_cm.png", "plots")
    plt.show()
    os.remove("final_cm.png")
    
    final_run_id = mlflow.active_run().info.run_id
    
    print(f"\n✅ Final model trained!")
    for k, v in metrics.items():
        print(f"   {k}: {v:.4f}")
    print(f"   Run ID: {final_run_id}")

## ➡️ Next: `04_model_comparison.ipynb` — Analyze and compare all results